In [2]:
import torch

x = torch.arange(4, dtype=torch.float, requires_grad=True)
y = 3 * torch.dot(x, x)
y.backward()
print(x.grad)
print(torch.allclose(x.grad, 6 * x))


tensor([ 0.,  6., 12., 18.])
True


In [5]:
x = torch.arange(4, dtype=torch.float, requires_grad=True)
y = 3 * torch.matmul(x, x)
y.backward()
print(x.grad)
print(torch.allclose(x.grad, 6 * x))


tensor([ 0.,  6., 12., 18.])
True


In [6]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = 2 * torch.dot(x, x)
y.backward()
print(x.grad)
# tensor([4.0, 8.0, 12.0])

tensor([ 4.,  8., 12.])


In [15]:
x = torch.tensor([1.0, 2.0], requires_grad=True)

y1 = (x * x).sum()
y1.backward()
print(x.grad)

y2 = x.sum()
y2.backward() #y2=x1+x2, 梯度应该是[1,1]

print(x.grad,'# 梯度没清理')


x.grad.zero_()
y2 = x.sum()
y2.backward() #y2=x1+x2, 梯度应该是[1,1]

print(x.grad, '正确结果')

tensor([2., 4.])
tensor([3., 5.]) # 梯度没清理
tensor([1., 1.]) 正确结果


In [17]:
# 求和函数的梯度，与向量x取值无关
x = torch.tensor([2.0, -1.0, 5.0], requires_grad=True)
y = x.sum()
y.backward()
x.grad

tensor([1., 1., 1.])

In [32]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x * x # y是向量，对应元素平方的向量，无法反向传播，报错，因此要求和
y.sum().backward() #y.sum()=x1**2+x2**2+x3**2
x.grad

tensor([2., 4., 6.])

In [33]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x * x # y是向量，对应元素平方的向量，无法反向传播，报错，因此可以引用一个 likey 的全1张量
y.backward(torch.ones_like(y))
x.grad

tensor([2., 4., 6.])

In [34]:
x = torch.tensor([2.0, 3.0], requires_grad=True)
y = x * x
u = y.detach()
z = u * x
z.sum().backward()
print(x.grad)


tensor([4., 9.])


In [35]:
x = torch.tensor([2.0, 3.0], requires_grad=True)
y = x * x
z = y.detach() * x
z.sum().backward()
x.grad #[4,9]

tensor([4., 9.])

In [36]:
x = torch.tensor([2.0, 3.0], requires_grad=True)
y = x * x
z = y * x
z.sum().backward()
x.grad #[12,27]

tensor([12., 27.])

In [46]:
x=torch.tensor([1.0],  requires_grad=True)
y=2*x
z=y**3
z.backward()
x.grad 

tensor([24.])

In [47]:
PyTorch 使用动态计算图。每次调用 f(a) 时，程序会根据当前输入实际执行一定次数的循环并选择一个分支。
PyTorch只记录本次运行中真正执行过的张量运算，并据此构建计算图。因此，即使不同输入产生不同的执行路径，PyTorch 仍可在本次构建的计算图上使用链式法则完成反向传播。
也就是说，即使是分段函数，也能够根据实际落在哪一段进行求导

SyntaxError: invalid character '。' (U+3002) (6820655.py, line 1)

In [48]:
x = torch.tensor(2.0, requires_grad=True)
y = 3 * x
z = y ** 2
z.backward()
x.grad, y.grad

C:\Users\11485\AppData\Local\Temp\ipykernel_20168\3363107373.py:5: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at C:\cb\pytorch_1000000000000\work\build\aten\src\ATen/core/TensorBody.h:494.)
  x.grad, y.grad


(tensor(36.), None)

In [50]:
# 报错因为y不是叶子张量，不会保留梯度，如果确实需要查看中间张量的梯度，可以在反向传播前调用：

x = torch.tensor(2.0, requires_grad=True)
y = 3 * x
z = y ** 2
y.retain_grad()
z.backward()
x.grad, y.grad

(tensor(36.), tensor(12.))

In [51]:
requires_grad_(True) 用于开启梯度跟踪，retain_grad() 用于保留非叶子张量的梯度。

SyntaxError: invalid character '，' (U+FF0C) (886880783.py, line 1)

In [54]:
#正向传播一次，构建一次计算图，执行反向传播一次，计算图摧毁
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
y.backward()
y.backward()



RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

In [56]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2
y.backward(retain_graph=True) #保留计算图，但会导致梯度累加
y.backward()
x.grad

tensor(8.)

In [60]:
x = torch.tensor(2.0, requires_grad=True)

y = x ** 2
y.backward()

x.grad.zero_()
y = x ** 2
y.backward()
x.grad

tensor(4.)

In [85]:
x = torch.tensor([1,2,3], dtype=torch.float, requires_grad=True)

z = 2*x**3 + 5*x
L=z.sum()
L.backward()
x.grad

tensor([11., 29., 59.])

In [87]:
# 15
x = torch.tensor([1.0, 2.0], requires_grad=True)

y = x ** 2
u = y.detach() #u=[1,4]
loss1 = (u * x).sum()
loss1.backward()

first_grad = x.grad.clone() #[1,4]
x.grad.zero_()

loss2 = (x ** 3).sum() 
loss2.backward()
second_grad = x.grad.clone() #[3,12]

first_grad, second_grad

(tensor([1., 4.]), tensor([ 3., 12.]))

In [96]:
import math
x = torch.tensor([1,2,3], dtype=torch.float, requires_grad=True)
y=torch.exp(x)
y.sum().backward()
x.grad==y

tensor([True, True, True])